# Bölüm: Spektral Diziler

Bu notebook **pytop** kütüphanesinin `spectral_sequences` modülünü kapsamlı biçimde ele almaktadır.
Spektral diziler, topoloji ve cebirsel geometride filtrelenmiş komplekslerin kohomolojisini
hesaplamak için kullanılan güçlü araçlardır.

**İçerik:**
1. Konu — Tanımlar, tablo ve içe aktarma
2. Teoremler — Temel teoremler ve doğrulama
3. Fonksiyon Analizi — 4 analiz fonksiyonu
4. API — Özet ve indeks fonksiyonları
5. Örnekler — Somut uygulamalar
6. Alıştırmalar — Egzersizler ve özet

## 1. Konu: Spektral Dizilerin Tanımı

Bir **spektral dizi** `{E_r^{p,q}, d_r}_{r≥2}` bir koleksiyondur; burada:
- `d_r : E_r^{p,q} → E_r^{p+r, q-r+1}` diferansiyelleri (kohomolojik konvansiyon)
- `d_r^2 = 0` koşulunu sağlar
- `E_{r+1}^{p,q} = ker(d_r) / im(d_r)` (sonraki sayfa)

### Temel Spektral Dizi Türleri

| Tür | E_2 Sayfası | Yakınsadığı | Birinci Çeyrek? |
|-----|-------------|-------------|------------------|
| Serre SS | `H^p(B; H^q(F))` | `H^*(E)` | Evet |
| Adams SS | `Ext_A(H^*(X), F_p)` | `π_*^s(X)_p` | Koşullu |
| Eilenberg-Moore SS | `Tor_{H^*(B)}(H^*(E), k)` | `H^*(fiber)` | Hayır |
| Atiyah-Hirzebruch SS | `H^p(X; h^q(pt))` | `h^*(X)` | Evet |
| Leray-Hirsch | `H^*(B) ⊗ H^*(F)` | `H^*(E)` | Evet |
| LHS SS | `H^p(Q; H^q(N; M))` | `H^*(G; M)` | Evet |
| Bockstein SS | `H^*(X; Z/p)` | `H^*(X; Z)` | Evet |
| Grothendieck SS | `R^pF(R^qG(A))` | `R^{p+q}(FG)(A)` | Evet |

### Filtrasyon ve Yakınsama

Güçlü yakınsama `E_r ⇒ H^n` şu anlama gelir:
- `H^n` üzerinde tam Hausdorff filtrasyonu `F^0 H^n ⊃ F^1 H^n ⊃ ... ⊃ 0`
- `E_∞^{p,q} = gr^p H^{p+q} = F^p H^{p+q} / F^{p+1} H^{p+q}`
- Birinci çeyrek spektral diziler her zaman güçlü yakınsar

In [ ]:
# İçe aktarma
from pytop.spectral_sequences import (
    SpectralSequenceProfile,
    SERRE_SS_TAGS, ADAMS_SS_TAGS, EILENBERG_MOORE_SS_TAGS,
    ATIYAH_HIRZEBRUCH_SS_TAGS, LERAY_SS_TAGS,
    CONVERGENCE_TAGS, DIFFERENTIAL_TAGS, FILTRATION_TAGS,
    get_named_spectral_sequence_profiles,
    spectral_sequence_layer_summary,
    spectral_sequence_chapter_index,
    spectral_sequence_type_index,
    is_multiplicative_spectral_sequence,
    converges_strongly,
    has_collapse_at_e2,
    is_first_quadrant_spectral_sequence,
    classify_spectral_sequence,
    spectral_sequence_profile,
)

print("Modül başarıyla yüklendi.")
print(f"Tag setleri: SERRE({len(SERRE_SS_TAGS)}), ADAMS({len(ADAMS_SS_TAGS)}),",
      f"EM({len(EILENBERG_MOORE_SS_TAGS)}), AHSS({len(ATIYAH_HIRZEBRUCH_SS_TAGS)}),",
      f"LERAY({len(LERAY_SS_TAGS)}), CONVERGENCE({len(CONVERGENCE_TAGS)})")

In [ ]:
# Tüm profilleri listele
profiles = get_named_spectral_sequence_profiles()
print(f"Toplam profil sayısı: {len(profiles)}\n")
for p in profiles:
    print(f"  [{p.key}]")
    print(f"    Tür: {p.sequence_type} | Kohomoloji: {p.cohomology_type}")
    print(f"    Yakınsadığı: {p.converges_to}")
    print(f"    Çarpımsal: {p.is_multiplicative} | 1.Çeyrek: {p.is_first_quadrant}",
          f"| E2'de çöküyor: {p.collapses_at_e2}")
    print()

In [ ]:
# Profil özellikleri — focus alanı örneği
serre_p = next(p for p in profiles if p.key == "serre_fibration_ss")
print(f"Profil: {serre_p.display_name}")
print(f"\nFocus özeti ({len(serre_p.focus)} karakter):")
print(serre_p.focus[:500] + "...")

adams_p = next(p for p in profiles if p.key == "adams_ss_stable_homotopy")
print(f"\nAdams SS profil (koşullu yakınsama={adams_p.is_conditionally_convergent}):")
print(adams_p.focus[:300] + "...")

## 2. Teoremler

### Teorem 1 (Serre Yakınsaması, 1951)
F → E → B bir fibrasyon ve B tek-bağlantılı ise, Serre spektral dizisi
`E_2^{p,q} = H^p(B; H^q(F)) ⇒ H^{p+q}(E)` şeklinde güçlü yakınsar.
Bu birinci çeyrek bir çarpımsal spektral dizidir; `d_r : E_r^{p,q} → E_r^{p+r, q-r+1}`
türevlemeleri sağlar: `d_r(xy) = d_r(x)y + (-1)^{|x|} x d_r(y)`.

### Teorem 2 (Adams E_2 Sayfası, 1958)
p-lokal bir spektrum X için Adams spektral dizisinin E_2 sayfası:
`E_2^{s,t} = Ext^{s,t}_A(H^*(X; F_p), F_p)` şeklindedir; burada A Steenrod cebiridir.
Bu, `π_{t-s}(X)_p^∧`'a (p-tamamlanmış stabil homotopi grupları) koşullu yakınsar.

### Teorem 3 (Leray-Hirsch Çöküşü)
F → E → B fibrasyon için H^*(F; k) serbest ve H^*(E; k)'dan F'ye küresel kesitler varsa
(Leray-Hirsch koşulu), Serre SS E_2'de çöküyor ve
`H^*(E; k) = H^*(B; k) ⊗_k H^*(F; k)` (H^*(B; k)-modüller olarak).

### Teorem 4 (Grothendieck SS, Türevli Fonktörlerin Bileşimi)
F: B→C, G: A→B sol-kesin fonktörler ve G enjektif nesneleri F-asiklik nesnelere
götürüyorsa, Grothendieck SS `E_2^{p,q} = (R^p F)(R^q G)(A) ⇒ R^{p+q}(FG)(A)` yakınsar.
Leray spektral dizisi `f: X→Y` ve `F = Γ(Y,-)`, `G = f_*` özel halidir.

In [ ]:
# Teoremleri doğrulama — profil özelliklerinden kontrol
def check_theorem(key, expected_fields):
    p = next(x for x in profiles if x.key == key)
    results = {}
    for field, expected in expected_fields.items():
        actual = getattr(p, field)
        results[field] = ("OK" if actual == expected else "FAIL", actual)
    return results

print("Teorem 1 — Serre SS güçlü yakınsar (is_conditionally_convergent=False):")
r = check_theorem("serre_fibration_ss", {"is_first_quadrant": True, "is_multiplicative": True,
                                          "is_conditionally_convergent": False})
for f, (st, v) in r.items(): print(f"  {f}: {st} ({v})")

print("\nTeorem 2 — Adams SS koşullu yakınsar:")
r = check_theorem("adams_ss_stable_homotopy", {"is_conditionally_convergent": True,
                                               "collapses_at_e2": False})
for f, (st, v) in r.items(): print(f"  {f}: {st} ({v})")

print("\nTeorem 3 — Leray-Hirsch E_2'de çöküyor:")
r = check_theorem("leray_hirsch_theorem", {"collapses_at_e2": True,
                                           "is_conditionally_convergent": False})
for f, (st, v) in r.items(): print(f"  {f}: {st} ({v})")

print("\nTeorem 4 — Grothendieck SS birinci çeyrekte:")
r = check_theorem("grothendieck_ss", {"is_first_quadrant": True,
                                      "is_conditionally_convergent": False})
for f, (st, v) in r.items(): print(f"  {f}: {st} ({v})")

## 3. Fonksiyon Analizi

Bu bölümde 4 analiz fonksiyonunun çok katmanlı karar mantığı incelenmektedir.
Her fonksiyon `Result` nesnesi döndürür; `r.is_true` / `r.is_false` ile sorgulanır.

### 3.1 `is_multiplicative_spectral_sequence(space)`

**Sözde kod:**
```
Katman 1: Açık çarpımsal etiket VEYA SERRE_SS_TAGS | ADAMS_SS_TAGS → true
Katman 2: ATIYAH_HIRZEBRUCH_SS_TAGS | LERAY_SS_TAGS → true
Katman 3: Bilinen çarpımsal olmayan (bockstein_ss, non_multiplicative_ss) → false
Katman 4: Bilinmiyor
```

**Karmaşıklık:** O(|tags|). **Avantaj:** 3 katman türü kapsar. **Sınırlama:** Yalnızca etiket tabanlı.

In [ ]:
class _S:
    def __init__(self, *tags): self.tags = set(tags)

print("=== is_multiplicative_spectral_sequence ===")
cases = [
    ("serre_ss", "Katman 1: Serre SS → true"),
    ("adams_spectral_sequence", "Katman 1: Adams SS → true"),
    ("ahss", "Katman 2: AHSS → true"),
    ("leray_hirsch", "Katman 2: Leray → true"),
    ("bockstein_ss", "Katman 3: Bockstein → false"),
    ("non_multiplicative_ss", "Katman 3: açık → false"),
]
for tag, desc in cases:
    r = is_multiplicative_spectral_sequence(_S(tag))
    status = "TRUE" if r.is_true else ("FALSE" if r.is_false else "UNKNOWN")
    print(f"  [{tag:30s}] → {status:8s} | {desc}")

r_unk = is_multiplicative_spectral_sequence(_S("tychonoff_space"))
print(f"  [tychonoff_space                ] → {'UNKNOWN':8s} | Katman 4: etiket yok")

### 3.2 `converges_strongly(space)`

**Sözde kod:**
```
Katman 1: strong_convergence | first_quadrant_ss → true
Katman 2: SERRE_SS_TAGS | AHSS_TAGS | LERAY_SS_TAGS | LHS etiketleri → true
Katman 3: conditional_convergence | ADAMS_SS_TAGS | EILENBERG_MOORE_SS_TAGS → false
Katman 4: Bilinmiyor
```

**Avantaj:** Birinci çeyrek SS'ler otomatik güçlü yakınsar. **Sınırlama:** lim^1 engeli tespit edilemez.

In [ ]:
print("=== converges_strongly ===")
cases = [
    ("strong_convergence",     "Katman 1: açık etiket → true"),
    ("first_quadrant_ss",      "Katman 1: birinci çeyrek → true"),
    ("serre_fibration",        "Katman 2: Serre türü → true"),
    ("atiyah_hirzebruch_ss",   "Katman 2: AHSS türü → true"),
    ("leray_ss",               "Katman 2: Leray türü → true"),
    ("lyndon_hochschild_serre","Katman 2: LHS türü → true"),
    ("adams_ss",               "Katman 3: Adams koşullu → false"),
    ("eilenberg_moore_ss",     "Katman 3: EM koşullu → false"),
    ("conditional_convergence","Katman 3: açık koşullu → false"),
]
for tag, desc in cases:
    r = converges_strongly(_S(tag))
    status = "TRUE" if r.is_true else ("FALSE" if r.is_false else "UNKNOWN")
    print(f"  [{tag:30s}] → {status:8s} | {desc}")

### 3.3 `has_collapse_at_e2(space)`

**Sözde kod:**
```
Katman 1: collapses_at_e2 | e2_degeneration | ... → true
Katman 2: leray_hirsch | trivial_fibration | bockstein_ss | LERAY_SS_TAGS → true
Katman 3: d3_differential | ADAMS_SS_TAGS | EILENBERG_MOORE_SS_TAGS → false
Katman 4: Bilinmiyor
```

**Not:** Leray-Hirsch koşulu E_2'de çöküşü garantiler; Adams ve EM SS'de yüksek diferansiyeller mevcuttur.

In [ ]:
print("=== has_collapse_at_e2 ===")
cases = [
    ("collapses_at_e2",     "Katman 1: açık etiket → true"),
    ("e2_degeneration",     "Katman 1: dejenerasyon → true"),
    ("leray_hirsch",        "Katman 2: Leray-Hirsch → true"),
    ("trivial_fibration",   "Katman 2: önemsiz fibrasyon → true"),
    ("bockstein_ss",        "Katman 2: Bockstein (p-torsyonsuz) → true"),
    ("leray_ss",            "Katman 2: Leray SS → true"),
    ("d3_differential",     "Katman 3: d_3 diferansiyeli var → false"),
    ("adams_ss",            "Katman 3: Adams SS → false"),
    ("eilenberg_moore_ss",  "Katman 3: EM SS → false"),
]
for tag, desc in cases:
    r = has_collapse_at_e2(_S(tag))
    status = "TRUE" if r.is_true else ("FALSE" if r.is_false else "UNKNOWN")
    print(f"  [{tag:30s}] → {status:8s} | {desc}")

### 3.4 `is_first_quadrant_spectral_sequence(space)`

**Sözde kod:**
```
Katman 1: first_quadrant_ss | strong_convergence → true
Katman 2: SERRE_SS_TAGS | AHSS_TAGS | LHS/Leray etiketleri → true
Katman 3: ADAMS_SS_TAGS | EILENBERG_MOORE_SS_TAGS → false (üst yarı / ikinci çeyrek)
Katman 4: Bilinmiyor
```

**Avantaj:** Adams SS üst yarı düzlem, EM SS ikinci çeyrek; bunlar birinci çeyrek değil.

In [ ]:
print("=== is_first_quadrant_spectral_sequence ===")
cases = [
    ("first_quadrant_ss",     "Katman 1: açık etiket → true"),
    ("serre_ss",              "Katman 2: Serre → true"),
    ("ahss",                  "Katman 2: AHSS → true"),
    ("grothendieck_ss",       "Katman 2: Grothendieck → true"),
    ("lhs_ss",                "Katman 2: LHS → true"),
    ("inflation_restriction", "Katman 2: LHS alt türü → true"),
    ("adams_ss",              "Katman 3: Adams (üst yarı düzlem) → false"),
    ("eilenberg_moore_ss",    "Katman 3: EM (ikinci çeyrek) → false"),
    ("loop_space_ss",         "Katman 3: EM alt türü → false"),
]
for tag, desc in cases:
    r = is_first_quadrant_spectral_sequence(_S(tag))
    status = "TRUE" if r.is_true else ("FALSE" if r.is_false else "UNKNOWN")
    print(f"  [{tag:30s}] → {status:8s} | {desc}")

## 4. API Referansı

| Fonksiyon | Açıklama | Dönüş Türü |
|-----------|----------|------------|
| `get_named_spectral_sequence_profiles()` | 8 kanonik profili döndürür | `tuple[SpectralSequenceProfile, ...]` |
| `spectral_sequence_layer_summary()` | Katman sayımı | `dict[str, int]` |
| `spectral_sequence_chapter_index()` | Bölüm → profil anahtarları | `dict[str, tuple[str,...]]` |
| `spectral_sequence_type_index()` | Tür → profil anahtarları | `dict[str, tuple[str,...]]` |
| `is_multiplicative_spectral_sequence(space)` | Çarpımsal mı? | `Result` |
| `converges_strongly(space)` | Güçlü yakınsar mı? | `Result` |
| `has_collapse_at_e2(space)` | E_2'de çöküyor mu? | `Result` |
| `is_first_quadrant_spectral_sequence(space)` | Birinci çeyrek mi? | `Result` |
| `classify_spectral_sequence(space)` | Tüm analizler | `dict[str, Result]` |
| `spectral_sequence_profile(space)` | Kapsamlı profil | `dict[str, Any]` |

In [ ]:
# Özet ve indeks fonksiyonları
print("Katman özeti:", spectral_sequence_layer_summary())
print()
print("Bölüm indeksi:")
for ch, keys in spectral_sequence_chapter_index().items():
    print(f"  Bölüm {ch}: {keys}")
print()
print("Tür indeksi:")
for t, keys in spectral_sequence_type_index().items():
    print(f"  {t}: {keys}")

## 5. Örnekler

Aşağıda 4 somut spektral dizi örneği üzerinde analizler yapılmaktadır.

In [ ]:
# Örnek 1: Serre spektral dizisi (path-loop fibrasyon)
# Omega X -> PX -> X, PX contractible
class SerrePathLoop:
    tags = {"serre_spectral_sequence", "serre_fibration", "path_space_fibration",
            "transgression", "edge_homomorphism", "first_quadrant_ss"}
    representation = "omega_x_path_loop_fibration"

r = classify_spectral_sequence(SerrePathLoop())
print("Örnek 1: Serre SS (Path-Loop Fibrasyon)")
print(f"  Çarpımsal: {r['is_multiplicative_spectral_sequence'].is_true}")
print(f"  Güçlü yakınsar: {r['converges_strongly'].is_true}")
print(f"  E_2'de çöküyor: {r['has_collapse_at_e2'].is_true}")
print(f"  Birinci çeyrek: {r['is_first_quadrant_spectral_sequence'].is_true}")
print(f"  Gerekçe: {r['is_multiplicative_spectral_sequence'].justification[0][:80]}...")

In [ ]:
# Örnek 2: Adams spektral dizisi (stabil homotopi grupları)
class AdamsStableHomotopy:
    tags = {"adams_spectral_sequence", "adams_ss", "ext_steenrod",
            "steenrod_algebra", "stable_homotopy", "hopf_invariant"}
    representation = "ext_a_f2_f2_stable_homotopy_spheres"

r = classify_spectral_sequence(AdamsStableHomotopy())
print("Örnek 2: Adams SS (p=2, Kürelerin Stabil Homotopi Grupları)")
print(f"  Çarpımsal: {r['is_multiplicative_spectral_sequence'].is_true}")
print(f"  Güçlü yakınsar: {r['converges_strongly'].is_true} (koşullu yakınsar)")
print(f"  E_2'de çöküyor: {r['has_collapse_at_e2'].is_true} (yüksek d_r mevcuttur)")
print(f"  Birinci çeyrek: {r['is_first_quadrant_spectral_sequence'].is_true} (üst yarı düzlem)")
prof = spectral_sequence_profile(AdamsStableHomotopy())
print(f"  Özet: {prof['summary']}")

In [ ]:
# Örnek 3: Leray-Hirsch (ürün fiber uzayı)
class LerayHirschFibration:
    tags = {"leray_hirsch", "product_fibration", "trivial_fibration",
            "global_sections_fiber", "strong_convergence"}
    representation = "f_times_b_trivial_bundle"

r = classify_spectral_sequence(LerayHirschFibration())
print("Örnek 3: Leray-Hirsch Teoremi (Trivial Fibrasyon)")
print(f"  Çarpımsal: {r['is_multiplicative_spectral_sequence'].is_true}")
print(f"  Güçlü yakınsar: {r['converges_strongly'].is_true}")
print(f"  E_2'de çöküyor: {r['has_collapse_at_e2'].is_true}")
print(f"  Birinci çeyrek: {r['is_first_quadrant_spectral_sequence'].is_true}")
print(f"  Gerekçe (çöküş): {r['has_collapse_at_e2'].justification[0][:100]}...")

In [ ]:
# Örnek 4: Eilenberg-Moore (loop uzayı kohomolojisi)
class EilenbergMooreLoopSpace:
    tags = {"eilenberg_moore_ss", "tor_functor_ss", "loop_space_ss",
            "bar_spectral_sequence", "path_fibration_ss"}
    representation = "tor_h_star_b_k_k_loop_b"

r = classify_spectral_sequence(EilenbergMooreLoopSpace())
print("Örnek 4: Eilenberg-Moore SS (Loop Uzayı H^*(Omega X))")
print(f"  Çarpımsal: {r['is_multiplicative_spectral_sequence'].is_true}")
print(f"  Güçlü yakınsar: {r['converges_strongly'].is_true}")
print(f"  E_2'de çöküyor: {r['has_collapse_at_e2'].is_true}")
print(f"  Birinci çeyrek: {r['is_first_quadrant_spectral_sequence'].is_true}")
print("  Not: EM SS ikinci çeyrektedir (p <= 0, q >= 0). Koşullu yakınsar.")

## 6. Alıştırmalar

Aşağıdaki alıştırmalar öğrencilere spektral dizi teorisini pekiştirmek için tasarlanmıştır.

**Alıştırma 1.** `S^1 → S^3 → S^2` Hopf fibrasyonu için Serre SS'i çizin. `H^*(S^2)` ve
`H^*(S^1)` kullanarak E_2 sayfasını yazın. Tek nonzero diferansiyelin ne olduğunu gösterin.

**Alıştırma 2.** `RP^∞ = K(Z/2, 1)` için `H^*(RP^∞; Z/2) = Z/2[x]` (|x|=1) sonucunu
Serre SS ile kanıtlayın: `S^∞ → RP^∞` fibrasyon olarak `Z/2 → S^∞ → RP^∞` alın.

**Alıştırma 3.** K-teorisi AHSS için `CP^2`'deki `d_3 = Sq^3`'ün sıfır olduğunu,
dolayısıyla `K^*(CP^2) ≅ Z^3` (derece 0) sonucunu elde edin.

**Alıştırma 4.** `1 → Z → Z → Z/n → 1` uzatması için LHS SS'in 5-terimli tam dizisini yazın
ve `H^1(Z/n; M)` ile `H^2(Z/n; M)`'yi hesaplayın (`M = Z`, trivial eylem).

**Alıştırma 5.** Bockstein SS'i `X = RP^2` için p=2'de hesaplayın.
`H^*(RP^2; Z) = {Z, 0, Z/2, 0, ...}` kullanarak E_∞ sayfasını belirleyin.

In [ ]:
# Alıştırma hazırlığı: tüm tag setleri
tag_sets = {
    "SERRE_SS_TAGS": SERRE_SS_TAGS,
    "ADAMS_SS_TAGS": ADAMS_SS_TAGS,
    "EILENBERG_MOORE_SS_TAGS": EILENBERG_MOORE_SS_TAGS,
    "ATIYAH_HIRZEBRUCH_SS_TAGS": ATIYAH_HIRZEBRUCH_SS_TAGS,
    "LERAY_SS_TAGS": LERAY_SS_TAGS,
    "CONVERGENCE_TAGS": CONVERGENCE_TAGS,
    "DIFFERENTIAL_TAGS": DIFFERENTIAL_TAGS,
    "FILTRATION_TAGS": FILTRATION_TAGS,
}
print("Kullanılabilir etiket setleri:")
for name, tagset in tag_sets.items():
    print(f"  {name} ({len(tagset)} etiket): {sorted(tagset)[:3]}...")

In [ ]:
# Öğrenci çalışma alanı: alıştırma 1 için yardımcı yapı
class HopfFibration:
    # S^1 -> S^3 -> S^2
    tags = {"serre_spectral_sequence", "gysin_sequence", "euler_class_ss"}
    representation = "hopf_s1_s3_s2"

r = classify_spectral_sequence(HopfFibration())
print("Hopf fibrasyonu için analiz:")
for key, result in r.items():
    status = "TRUE" if result.is_true else ("FALSE" if result.is_false else "UNKNOWN")
    print(f"  {key}: {status}")
    if result.is_true or result.is_false:
        print(f"    Gerekçe: {result.justification[0][:80]}...")

## Özet

Bu notebook `pytop.spectral_sequences` modülünü kapsamlı biçimde ele aldı:

- **8 profil:** Serre, Adams, Eilenberg-Moore, Atiyah-Hirzebruch, Leray-Hirsch, LHS, Bockstein, Grothendieck
- **8 tag frozenset:** `SERRE_SS_TAGS`, `ADAMS_SS_TAGS`, `EILENBERG_MOORE_SS_TAGS`,
  `ATIYAH_HIRZEBRUCH_SS_TAGS`, `LERAY_SS_TAGS`, `CONVERGENCE_TAGS`, `DIFFERENTIAL_TAGS`, `FILTRATION_TAGS`
- **4 analiz fonksiyonu:** `is_multiplicative_spectral_sequence`, `converges_strongly`,
  `has_collapse_at_e2`, `is_first_quadrant_spectral_sequence`
- **2 cephe fonksiyonu:** `classify_spectral_sequence`, `spectral_sequence_profile`

**Temel farklar:**
- Birinci çeyrek SS (Serre, AHSS, Leray, LHS, Grothendieck): her zaman güçlü yakınsar
- Adams SS: üst yarı düzlem, koşullu yakınsama, yüksek diferansiyeller
- Eilenberg-Moore SS: ikinci çeyrek, koşullu yakınsama
- Leray-Hirsch koşulu: E_2'de çöküşü garantiler